In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("churn.db")

tables = pd.read_sql("SELECT name, type FROM sqlite_master WHERE type IN ('table','view');", conn)
print(tables)

print(pd.read_sql("SELECT * FROM customers LIMIT 5;", conn))

                      name   type
0                customers  table
1         segment_contract  table
2      segment_techsupport  table
3  segment_internetservice  table
4    segment_paymentmethod  table
5      risk_score_baseline   view
6          customer_master   view
7     customer_risk_scores  table
8    retention_action_list  table
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No        

In [2]:
overall = pd.read_sql("""
    SELECT Churn, COUNT(*) as customers,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customers), 1) as pct
    FROM customers
    GROUP BY Churn;
""", conn)
print(overall)

  Churn  customers   pct
0    No       5174  73.5
1   Yes       1869  26.5


In [3]:
contract_df = pd.read_sql("""
    SELECT Contract,
           COUNT(*) as total_customers,
           SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) as churned,
           ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) as churn_rate_pct
    FROM customers
    GROUP BY Contract
    ORDER BY churn_rate_pct DESC;
""", conn)
print(contract_df)
contract_df.to_sql("segment_contract", conn, if_exists="replace", index=False)

         Contract  total_customers  churned  churn_rate_pct
0  Month-to-month             3875     1655            42.7
1        One year             1473      166            11.3
2        Two year             1695       48             2.8


3

In [4]:
segments = {
    "TechSupport": """
        SELECT TechSupport, COUNT(*) total,
               SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) churned,
               ROUND(100.0*SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END)/COUNT(*),1) churn_rate
        FROM customers GROUP BY TechSupport
    """,
    "InternetService": """
        SELECT InternetService, COUNT(*) total,
               SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) churned,
               ROUND(100.0*SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END)/COUNT(*),1) churn_rate
        FROM customers GROUP BY InternetService
    """,
    "PaymentMethod": """
        SELECT PaymentMethod, COUNT(*) total,
               SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) churned,
               ROUND(100.0*SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END)/COUNT(*),1) churn_rate
        FROM customers GROUP BY PaymentMethod
    """,
}

for name, q in segments.items():
    result = pd.read_sql(q, conn)
    print(f"\n--- {name} ---")
    print(result)
    result.to_sql(f"segment_{name.lower()}", conn, if_exists="replace", index=False)


--- TechSupport ---
           TechSupport  total  churned  churn_rate
0                   No   3473     1446        41.6
1  No internet service   1526      113         7.4
2                  Yes   2044      310        15.2

--- InternetService ---
  InternetService  total  churned  churn_rate
0             DSL   2421      459        19.0
1     Fiber optic   3096     1297        41.9
2              No   1526      113         7.4

--- PaymentMethod ---
               PaymentMethod  total  churned  churn_rate
0  Bank transfer (automatic)   1544      258        16.7
1    Credit card (automatic)   1522      232        15.2
2           Electronic check   2365     1071        45.3
3               Mailed check   1612      308        19.1


In [7]:
tenure_df = pd.read_sql("""
    SELECT
        CASE
            WHEN tenure <= 12 THEN '0-1 year'
            WHEN tenure <= 24 THEN '1-2 years'
            WHEN tenure <= 48 THEN '2-4 years'
            ELSE '4+ years'
        END AS tenure_bucket,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_rate_pct
    FROM customers
    GROUP BY tenure_bucket
    ORDER BY churn_rate_pct DESC;
""", conn)
print(tenure_df)
tenure_df.to_sql("segment_tenure", conn, if_exists="replace", index=False)

  tenure_bucket  total_customers  churned  churn_rate_pct
0      0-1 year             2186     1037            47.4
1     1-2 years             1024      294            28.7
2     2-4 years             1594      325            20.4
3      4+ years             2239      213             9.5


4

In [8]:
charge_df = pd.read_sql("""
    SELECT
        CASE
            WHEN MonthlyCharges < 35 THEN 'Low ($0-35)'
            WHEN MonthlyCharges < 70 THEN 'Medium ($35-70)'
            WHEN MonthlyCharges < 90 THEN 'High ($70-90)'
            ELSE 'Very High ($90+)'
        END AS charge_bucket,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_rate_pct
    FROM customers
    GROUP BY charge_bucket
    ORDER BY churn_rate_pct DESC;
""", conn)
print(charge_df)
charge_df.to_sql("segment_charges", conn, if_exists="replace", index=False)

      charge_bucket  total_customers  churned  churn_rate_pct
0     High ($70-90)             1847      701            38.0
1  Very High ($90+)             1744      573            32.9
2   Medium ($35-70)             1721      407            23.6
3       Low ($0-35)             1731      188            10.9


4

In [9]:
combined_df = pd.read_sql("""
    SELECT
        Contract, InternetService, TechSupport,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_rate_pct
    FROM customers
    GROUP BY Contract, InternetService, TechSupport
    HAVING COUNT(*) >= 30
    ORDER BY churn_rate_pct DESC
    LIMIT 15;
""", conn)
print(combined_df)

          Contract InternetService          TechSupport  total_customers  \
0   Month-to-month     Fiber optic                   No             1796   
1   Month-to-month     Fiber optic                  Yes              332   
2   Month-to-month             DSL                   No              884   
3   Month-to-month             DSL                  Yes              339   
4         One year     Fiber optic                  Yes              226   
5   Month-to-month              No  No internet service              524   
6         One year     Fiber optic                   No              313   
7         One year             DSL                   No              244   
8         One year             DSL                  Yes              326   
9         Two year     Fiber optic                   No              121   
10        Two year     Fiber optic                  Yes              308   
11        Two year             DSL                   No              115   
12        On

In [10]:
revenue_df = pd.read_sql("""
    SELECT Contract,
           COUNT(*) AS churned_customers,
           ROUND(SUM(MonthlyCharges), 2) AS monthly_revenue_lost,
           ROUND(SUM(MonthlyCharges) * 12, 2) AS annual_revenue_lost
    FROM customers
    WHERE Churn = 'Yes'
    GROUP BY Contract
    ORDER BY annual_revenue_lost DESC;
""", conn)
print(revenue_df)

         Contract  churned_customers  monthly_revenue_lost  \
0  Month-to-month               1655             120847.10   
1        One year                166              14118.45   
2        Two year                 48               4165.30   

   annual_revenue_lost  
0            1450165.2  
1             169421.4  
2              49983.6  


In [11]:
cursor = conn.cursor()
cursor.executescript("""
DROP VIEW IF EXISTS risk_score_baseline;
CREATE VIEW risk_score_baseline AS
SELECT
    customerID, Contract, tenure, MonthlyCharges, TechSupport, InternetService, Churn,
    (
        CASE WHEN Contract = 'Month-to-month' THEN 3 ELSE 0 END +
        CASE WHEN tenure <= 12 THEN 2 ELSE 0 END +
        CASE WHEN TechSupport = 'No' THEN 1 ELSE 0 END +
        CASE WHEN InternetService = 'Fiber optic' THEN 1 ELSE 0 END +
        CASE WHEN MonthlyCharges >= 70 THEN 1 ELSE 0 END
    ) AS rule_risk_score
FROM customers;

DROP VIEW IF EXISTS customer_master;
CREATE VIEW customer_master AS
SELECT c.*, r.rule_risk_score
FROM customers c
JOIN risk_score_baseline r ON c.customerID = r.customerID;
""")
conn.commit()

print(pd.read_sql("SELECT * FROM customer_master LIMIT 5;", conn))

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... TechSupport  \
0  No phone service             DSL             No  ...          No   
1                No             DSL            Yes  ...          No   
2                No             DSL            Yes  ...          No   
3  No phone service             DSL            Yes  ...         Yes   
4                No     Fiber optic             No  ...          No   

  StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No             

In [14]:
validate_df = pd.read_sql("""
    SELECT
        rule_risk_score,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS actual_churn_rate
    FROM risk_score_baseline
    GROUP BY rule_risk_score
    ORDER BY rule_risk_score DESC;
""", conn)
print(validate_df)

conn.close()

ProgrammingError: Cannot operate on a closed database.